In [1]:
# CELL 1 — Imports

import torch
import torch.nn as nn
import torchvision.models as models
import timm
import torchvision.transforms as transforms
from torchvision import datasets
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import joblib

print("✅ All imports done")

✅ All imports done


In [2]:
# CELL 2 — Load all trained models

from pathlib import Path

# Repo root: walk up from the notebook's directory until app.py is found.
BASE_PATH    = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "app.py").is_file())
KAGGLE_TRAIN = BASE_PATH / "datasets" / "KAGGLE DATASET" / "Combined Dataset" / "train"
MODELS_PATH  = BASE_PATH / "models"

device = torch.device('cpu')  # CPU for stability
print(f"Using device: {device}")

# Define model classes
class FusionLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(13, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(16, 2)
        )
    
    def forward(self, x):
        return self.fc(x)

# Load class names
dataset_raw = datasets.ImageFolder(KAGGLE_TRAIN, transform=None)
class_names = dataset_raw.classes
print(f"Classes: {class_names}")

print("\n✅ Models loaded successfully")

Using device: cpu
Classes: ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']

✅ Models loaded successfully


In [5]:
# CELL 3 — Load Final Models

def build_resnet152():
    model = models.resnet152(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(2048, 4)
    )
    return model


def build_vgg16():
    model = models.vgg16(weights=None)
    model.classifier[6] = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(4096, 4)
    )
    return model


def build_efficientnet():
    return timm.create_model(
        "efficientnet_b4",
        pretrained=False,
        num_classes=4
    )


def build_vit():
    return timm.create_model(
        "vit_base_patch16_224",
        pretrained=False,
        num_classes=4
    )


print("Loading final models...")

resnet = build_resnet152()
vgg = build_vgg16()
efficient = build_efficientnet()
vit = build_vit()


resnet.load_state_dict(
    torch.load(
        os.path.join(
            MODELS_PATH,
            "resnet152_best.pth"
        ),
        map_location=device
    )
)

vgg.load_state_dict(
    torch.load(
        os.path.join(
            MODELS_PATH,
            "vgg16_best.pth"
        ),
        map_location=device
    )
)

efficient.load_state_dict(
    torch.load(
        os.path.join(
            MODELS_PATH,
            "efficientnet_b4_best.pth"
        ),
        map_location=device
    )
)

vit.load_state_dict(
    torch.load(
        os.path.join(
            MODELS_PATH,
            "vit_best.pth"
        ),
        map_location=device
    )

)
meta_learner = joblib.load(
    os.path.join(
        MODELS_PATH,
        "ensemble_meta_learner.pkl"
    )
)


all_models = [
    resnet,
    vgg,
    efficient,
    vit
]

for model in all_models:
    model.to(device)
    model.eval()


print("ResNet-152 loaded")
print("VGG16 loaded")
print("EfficientNet-B4 loaded")
print("ViT loaded")
print("Ensemble meta-learner loaded")

print("\n" + "=" * 60)
print("ALL FINAL ENSEMBLE MODELS LOADED")
print("=" * 60)

Loading final models...
ResNet-152 loaded
VGG16 loaded
EfficientNet-B4 loaded
ViT loaded
Ensemble meta-learner loaded

ALL FINAL ENSEMBLE MODELS LOADED


In [10]:
# CELL 4A — Clinical Branch Definition

class ClinicalFusionNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.network(x)


print("ClinicalFusionNet defined")

ClinicalFusionNet defined


In [14]:
# CELL 4 — Final Prediction Function

CLASSES = [
    "Mild Impairment",
    "Moderate Impairment",
    "No Impairment",
    "Very Mild Impairment"
]

clinical_model = ClinicalFusionNet().to(device)

clinical_model.load_state_dict(
    torch.load(
        os.path.join(
            MODELS_PATH,
            "clinical_branch_best.pth"
        ),
        map_location=device
    )
)

clinical_model.eval()


transform_224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

transform_380 = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def predict_alzheimers(
    image_path,
    mmse_score=24,
    cdr_score=0.5,
    nwbv_score=0.73
):

    print("\n" + "=" * 60)
    print("ALZHEIMER'S DISEASE PREDICTION")
    print("=" * 60)

    print(f"\nImage: {image_path}")

    image = Image.open(image_path)

    image_224 = transform_224(
        image
    ).unsqueeze(0).to(device)

    image_380 = transform_380(
        image
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        out_resnet = torch.softmax(
            resnet(image_224), dim=1
        )[0].cpu().numpy()

        out_vgg = torch.softmax(
            vgg(image_224), dim=1
        )[0].cpu().numpy()

        out_eff = torch.softmax(
            efficient(image_380), dim=1
        )[0].cpu().numpy()

        out_vit = torch.softmax(
            vit(image_224), dim=1
        )[0].cpu().numpy()

    outputs = [
        out_resnet,
        out_vgg,
        out_eff,
        out_vit
    ]

    names = [
        "ResNet-152",
        "VGG16",
        "EfficientNet-B4",
        "ViT"
    ]

    print("\nMRI Models:")
    print("-" * 60)

    for name, output in zip(names, outputs):

        prediction = output.argmax()

        print(
            f"{name:<20}: "
            f"{CLASSES[prediction]:<25} "
            f"{output.max() * 100:.2f}%"
        )

    # 16 features: 4 probabilities × 4 models
    features = np.hstack(outputs).reshape(1, -1)

    ensemble_prediction = meta_learner.predict(
        features
    )[0]

    ensemble_probabilities = (
        meta_learner.predict_proba(features)[0]
    )

    ensemble_confidence = (
        ensemble_probabilities.max() * 100
    )

    mri_class = CLASSES[ensemble_prediction]

    print("\nMRI Ensemble:")
    print("-" * 60)
    print(f"Prediction : {mri_class}")
    print(f"Confidence : {ensemble_confidence:.2f}%")

    # Clinical branch
    mmse_norm = mmse_score / 30.0

    cdr_norm = cdr_scaler.transform(
        [[cdr_score]]
    )[0][0]

    nwbv_norm = nwbv_scaler.transform(
        [[nwbv_score]]
    )[0][0]

    clinical_input = np.array(
        [
            mmse_norm,
            cdr_norm,
            nwbv_norm
        ],
        dtype=np.float32
    )

    clinical_tensor = torch.tensor(
        clinical_input
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        clinical_output = clinical_model(
            clinical_tensor
        )

        clinical_probabilities = torch.softmax(
            clinical_output,
            dim=1
        )[0].cpu().numpy()

    clinical_prediction = clinical_output.argmax(
        dim=1
    ).item()

    clinical_class = (
        "DEMENTED"
        if clinical_prediction == 0
        else "NON-DEMENTED"
    )

    clinical_confidence = (
        clinical_probabilities.max() * 100
    )

    print("\nClinical Branch:")
    print("-" * 60)
    print(f"Prediction : {clinical_class}")
    print(f"Confidence : {clinical_confidence:.2f}%")

    print("\nClinical inputs:")
    print(f"MMSE : {mmse_score}")
    print(f"CDR  : {cdr_score}")
    print(f"nWBV : {nwbv_score}")

    print("\n" + "=" * 60)
    print("PREDICTION COMPLETE")
    print("=" * 60)

    return {
        "mri_class": mri_class,
        "mri_confidence": ensemble_confidence,
        "ensemble_probabilities": ensemble_probabilities,
        "clinical_class": clinical_class,
        "clinical_confidence": clinical_confidence,
        "clinical_probabilities": clinical_probabilities,
        "resnet_prediction": CLASSES[out_resnet.argmax()],
        "vgg_prediction": CLASSES[out_vgg.argmax()],
        "efficientnet_prediction": CLASSES[out_eff.argmax()],
        "vit_prediction": CLASSES[out_vit.argmax()]
    }


print("Prediction function ready")

Prediction function ready


In [16]:
# CELL 6 — Clinical Preprocessing

import pandas as pd
from sklearn.preprocessing import StandardScaler

clinical_data = pd.read_csv(
    os.path.join(
        BASE_PATH,
        "datasets",
        "alzheimer.csv"
    )
)

cdr_scaler = StandardScaler()
cdr_scaler.fit(
    clinical_data[["CDR"]].fillna(
        clinical_data["CDR"].median()
    )
)

nwbv_scaler = StandardScaler()
nwbv_scaler.fit(
    clinical_data[["nWBV"]].fillna(
        clinical_data["nWBV"].median()
    )
)

print("Clinical preprocessing ready")

Clinical preprocessing ready


In [17]:
# CELL 5 — Test Images

print("Finding test images...")

test_samples = {}

for class_name in CLASSES:

    class_path = os.path.join(
        KAGGLE_TRAIN,
        class_name
    )

    images = [
        f for f in os.listdir(class_path)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ]

    if images:
        test_samples[class_name] = os.path.join(
            class_path,
            images[0]
        )

print("\n" + "=" * 60)
print("RUNNING FINAL MODEL PREDICTIONS")
print("=" * 60)

results = {}

for actual_class, image_path in test_samples.items():

    print(f"\nActual class: {actual_class}")

    result = predict_alzheimers(
        image_path,
        mmse_score=22,
        cdr_score=0.5,
        nwbv_score=0.73
    )

    results[actual_class] = result

    print("\nResult:")
    print(
        f"MRI prediction : "
        f"{result['mri_class']}"
    )

    print(
        f"MRI confidence : "
        f"{result['mri_confidence']:.2f}%"
    )

    print(
        f"Clinical result : "
        f"{result['clinical_class']}"
    )

    print(
        f"Clinical confidence : "
        f"{result['clinical_confidence']:.2f}%"
    )

Finding test images...

RUNNING FINAL MODEL PREDICTIONS

Actual class: Mild Impairment

ALZHEIMER'S DISEASE PREDICTION

Image: datasets\KAGGLE DATASET\Combined Dataset\train\Mild Impairment\MildImpairment (1).jpg

MRI Models:
------------------------------------------------------------
ResNet-152          : Mild Impairment           100.00%
VGG16               : Very Mild Impairment      46.04%
EfficientNet-B4     : Mild Impairment           100.00%
ViT                 : Mild Impairment           100.00%

MRI Ensemble:
------------------------------------------------------------
Prediction : Mild Impairment
Confidence : 99.40%

Clinical Branch:
------------------------------------------------------------
Prediction : DEMENTED
Confidence : 97.76%

Clinical inputs:
MMSE : 22
CDR  : 0.5
nWBV : 0.73

PREDICTION COMPLETE

Result:
MRI prediction : Mild Impairment
MRI confidence : 99.40%
Clinical result : DEMENTED
Clinical confidence : 97.76%

Actual class: Moderate Impairment

ALZHEIMER'S DI

site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



MRI Models:
------------------------------------------------------------
ResNet-152          : Moderate Impairment       100.00%
VGG16               : Very Mild Impairment      36.99%
EfficientNet-B4     : Moderate Impairment       100.00%
ViT                 : Moderate Impairment       100.00%

MRI Ensemble:
------------------------------------------------------------
Prediction : Moderate Impairment
Confidence : 97.29%

Clinical Branch:
------------------------------------------------------------
Prediction : DEMENTED
Confidence : 97.76%

Clinical inputs:
MMSE : 22
CDR  : 0.5
nWBV : 0.73

PREDICTION COMPLETE

Result:
MRI prediction : Moderate Impairment
MRI confidence : 97.29%
Clinical result : DEMENTED
Clinical confidence : 97.76%

Actual class: No Impairment

ALZHEIMER'S DISEASE PREDICTION

Image: datasets\KAGGLE DATASET\Combined Dataset\train\No Impairment\NoImpairment (1).jpg


site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



MRI Models:
------------------------------------------------------------
ResNet-152          : No Impairment             99.95%
VGG16               : Very Mild Impairment      53.04%
EfficientNet-B4     : No Impairment             100.00%
ViT                 : No Impairment             98.36%

MRI Ensemble:
------------------------------------------------------------
Prediction : No Impairment
Confidence : 98.72%

Clinical Branch:
------------------------------------------------------------
Prediction : DEMENTED
Confidence : 97.76%

Clinical inputs:
MMSE : 22
CDR  : 0.5
nWBV : 0.73

PREDICTION COMPLETE

Result:
MRI prediction : No Impairment
MRI confidence : 98.72%
Clinical result : DEMENTED
Clinical confidence : 97.76%

Actual class: Very Mild Impairment

ALZHEIMER'S DISEASE PREDICTION

Image: datasets\KAGGLE DATASET\Combined Dataset\train\Very Mild Impairment\VeryMildImpairment (1).jpg


site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



MRI Models:
------------------------------------------------------------
ResNet-152          : Very Mild Impairment      99.79%
VGG16               : No Impairment             76.85%
EfficientNet-B4     : Very Mild Impairment      98.74%
ViT                 : No Impairment             99.52%

MRI Ensemble:
------------------------------------------------------------
Prediction : Very Mild Impairment
Confidence : 94.36%

Clinical Branch:
------------------------------------------------------------
Prediction : DEMENTED
Confidence : 97.76%

Clinical inputs:
MMSE : 22
CDR  : 0.5
nWBV : 0.73

PREDICTION COMPLETE

Result:
MRI prediction : Very Mild Impairment
MRI confidence : 94.36%
Clinical result : DEMENTED
Clinical confidence : 97.76%


site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [18]:
# CELL 6 — Summary

print("\n" + "=" * 60)
print("PREDICTION SUMMARY")
print("=" * 60)

for actual_class, result in results.items():

    print(f"\nActual class       : {actual_class}")
    print(f"MRI prediction     : {result['mri_class']}")
    print(
        f"MRI confidence     : "
        f"{result['mri_confidence']:.2f}%"
    )
    print(f"Clinical prediction: {result['clinical_class']}")
    print(
        f"Clinical confidence: "
        f"{result['clinical_confidence']:.2f}%"
    )

print("\n" + "=" * 60)
print("SYSTEM READY")
print("=" * 60)


PREDICTION SUMMARY

Actual class       : Mild Impairment
MRI prediction     : Mild Impairment
MRI confidence     : 99.40%
Clinical prediction: DEMENTED
Clinical confidence: 97.76%

Actual class       : Moderate Impairment
MRI prediction     : Moderate Impairment
MRI confidence     : 97.29%
Clinical prediction: DEMENTED
Clinical confidence: 97.76%

Actual class       : No Impairment
MRI prediction     : No Impairment
MRI confidence     : 98.72%
Clinical prediction: DEMENTED
Clinical confidence: 97.76%

Actual class       : Very Mild Impairment
MRI prediction     : Very Mild Impairment
MRI confidence     : 94.36%
Clinical prediction: DEMENTED
Clinical confidence: 97.76%

SYSTEM READY
